In [17]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-03-25 01:04:53 +05:30)


## Model Training Pipeline

The complete workflow of the model training process is:

1. Load dataset (Normal + Pneumonia images)
2. Preprocess images (resize, normalize, augment)
3. Create data generators
4. Split data (Normal vs Pneumonia)
5. Apply training strategy:
   - Either standard training
   - Or 3-Fold cross validation
6. Train CNN model
8. Save model

In [2]:
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.optimizers import Adam

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Standard Data Flow (Without Folding)

In the normal training approach:

- Dataset is divided into:
  - Training Set
  - Validation Set
  - Test Set

### Flow:

Dataset → Train Generator → Model → Validation Generator → Metrics

### Explanation:

1. Training generator feeds batches of images to the model.
2. Model learns patterns (Normal vs Pneumonia).
3. Validation generator checks performance after each epoch.
4. Test set is used only after training is complete.

In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

In [6]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

We aren't using advanced data augmentation because it will risk the data. problems:-
- large data
- brightness and contrast matters

In [7]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

A generator does NOT load all images at once.

Instead:
- It loads images in batches (e.g., 32 images)
- Applies preprocessing (resize, normalize)
- Feeds them to the model

Why needed?

- Saves memory
- Enables large dataset training
- Allows augmentation

In [8]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

Found 5216 images belonging to 2 classes.


In [9]:
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False
)

Found 16 images belonging to 2 classes.


In [10]:
labels = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))

print("Class indices:", train_generator.class_indices)
print("Class distribution:", np.bincount(labels))
print("Class weights:", class_weights)

Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}
Class distribution: [1341 3875]
Class weights: {0: np.float64(1.9448173005219984), 1: np.float64(0.6730322580645162)}


The solution we used: Class Weights
* If a class has fewer images → its mistakes are more costly
* If a class has many images → its mistakes cost less

In [11]:
from tensorflow.keras.optimizers import Adam

def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return m

layers.GlobalAveragePooling2D() :-
To reduce parameters and prevent overfitting by summarizing each feature map instead of flattening everything

In [12]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_model.h5",
        monitor="val_loss",
        save_best_only=True
    )
]

In [13]:
model = build_model()
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 873ms/step - accuracy: 0.7803 - loss: 0.4473

163/163 ━━━━━━━━━━━━━━━━━━━━ 148s 878ms/step - accuracy: 0.8388 - loss: 0.3588 - val_accuracy: 0.5000 - val_loss: 2.9301
Epoch 2/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 136s 830ms/step - accuracy: 0.8936 - loss: 0.2460 - val_accuracy: 0.5000 - val_loss: 4.2002
Epoch 3/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 136s 832ms/step - accuracy: 0.8999 - loss: 0.2365 - val_accuracy: 0.5000 - val_loss: 4.6113
Epoch 4/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 946ms/step - accuracy: 0.9150 - loss: 0.1971

163/163 ━━━━━━━━━━━━━━━━━━━━ 155s 949ms/step - accuracy: 0.9166 - loss: 0.1995 - val_accuracy: 0.5625 - val_loss: 1.6427
Epoch 5/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 902ms/step - accuracy: 0.9231 - loss: 0.1814

163/163 ━━━━━━━━━━━━━━━━━━━━ 195s 904ms/step - accuracy: 0.9285 - loss: 0.1753 - val_accuracy: 0.6250 - val_loss: 1.3501
Epoch 6/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 141s 864ms/step - accuracy: 0.9316 - loss: 0.1737 - val_accuracy: 0.5000 - val_loss: 4.1834
Epoch 7/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 140s 859ms/step - accuracy: 0.9390 - loss: 0.1587 - val_accuracy: 0.5000 - val_loss: 2.3170
Epoch 8/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - accuracy: 0.9362 - loss: 0.1580

163/163 ━━━━━━━━━━━━━━━━━━━━ 140s 855ms/step - accuracy: 0.9392 - loss: 0.1533 - val_accuracy: 0.5000 - val_loss: 1.0889
Epoch 9/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 863ms/step - accuracy: 0.9408 - loss: 0.1416

163/163 ━━━━━━━━━━━━━━━━━━━━ 141s 865ms/step - accuracy: 0.9390 - loss: 0.1499 - val_accuracy: 0.7500 - val_loss: 0.7358
Epoch 10/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 140s 855ms/step - accuracy: 0.9406 - loss: 0.1481 - val_accuracy: 0.5000 - val_loss: 1.6049
Epoch 11/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 840ms/step - accuracy: 0.9410 - loss: 0.1500

163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 842ms/step - accuracy: 0.9434 - loss: 0.1471 - val_accuracy: 0.6875 - val_loss: 0.6659
Epoch 12/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 137s 839ms/step - accuracy: 0.9500 - loss: 0.1294 - val_accuracy: 0.4375 - val_loss: 1.3015
Epoch 13/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 843ms/step - accuracy: 0.9511 - loss: 0.1233 - val_accuracy: 0.5000 - val_loss: 1.4788
Epoch 14/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 843ms/step - accuracy: 0.9505 - loss: 0.1233 - val_accuracy: 0.5000 - val_loss: 1.5755
Epoch 15/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 831ms/step - accuracy: 0.9512 - loss: 0.1324

163/163 ━━━━━━━━━━━━━━━━━━━━ 136s 833ms/step - accuracy: 0.9569 - loss: 0.1172 - val_accuracy: 0.7500 - val_loss: 0.6137
Epoch 16/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 842ms/step - accuracy: 0.9549 - loss: 0.1148 - val_accuracy: 0.5000 - val_loss: 1.1572
Epoch 17/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 142s 869ms/step - accuracy: 0.9572 - loss: 0.1094 - val_accuracy: 0.5000 - val_loss: 1.9582
Epoch 18/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 898ms/step - accuracy: 0.9435 - loss: 0.1299

163/163 ━━━━━━━━━━━━━━━━━━━━ 147s 900ms/step - accuracy: 0.9500 - loss: 0.1150 - val_accuracy: 0.8125 - val_loss: 0.5320
Epoch 19/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 845ms/step - accuracy: 0.9567 - loss: 0.1115

163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 847ms/step - accuracy: 0.9563 - loss: 0.1139 - val_accuracy: 0.8125 - val_loss: 0.3830
Epoch 20/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 139s 851ms/step - accuracy: 0.9590 - loss: 0.1051 - val_accuracy: 0.6250 - val_loss: 1.2119
Epoch 21/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 137s 836ms/step - accuracy: 0.9528 - loss: 0.1191 - val_accuracy: 0.7500 - val_loss: 0.5999
Epoch 22/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 137s 837ms/step - accuracy: 0.9580 - loss: 0.1033 - val_accuracy: 0.6875 - val_loss: 0.7394
Epoch 23/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 137s 838ms/step - accuracy: 0.9597 - loss: 0.1069 - val_accuracy: 0.7500 - val_loss: 0.4891
Epoch 24/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 841ms/step - accuracy: 0.9655 - loss: 0.0962 - val_accuracy: 0.6250 - val_loss: 0.7432


### What’s the issue now?

Right now the result is from **one train/val split**.
That means:
* It could be **lucky**
* It could be **biased to that split**
* It might **drop on another split**
So your model is *working*, but not yet *reliable*

### Why move to K-Fold?
K-Fold answers one question:
> “Does my model perform consistently on different data splits?”
### With K-Fold:
* Multiple accuracies (e.g. 0.82, 0.87, 0.84)
* You get:
  * Mean performance
  * Variance (very important)

### Bottom line

* Fixed **how the model learns** 
* Now fix **how you validate it** → K-Fold

If the K-Fold scores stay consistent → your model is genuinely strong
If they fluctuate → you still have instability to fix


In [14]:
all_paths = []
all_labels = []

class_map = train_generator.class_indices

for class_name, label in class_map.items():
    class_dir = TRAIN_DIR / class_name
    
    for img_name in os.listdir(class_dir):
        all_paths.append(str(class_dir / img_name))
        all_labels.append(label)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

print("Total samples:", len(all_paths))
print("Class distribution:", np.bincount(all_labels))

Total samples: 5216
Class distribution: [1341 3875]


In [15]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_histories = []
fold_val_accuracies = []

In [18]:
for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels), 1):
    
    print(f"\n------------Fold {fold}------------")
    
    train_paths, val_paths = all_paths[train_idx], all_paths[val_idx]
    train_labels, val_labels = all_labels[train_idx], all_labels[val_idx]
        
    train_df = pd.DataFrame({
        "filename": train_paths,
        "class": train_labels
    })
    
    val_df = pd.DataFrame({
        "filename": val_paths,
        "class": val_labels
    })
    
    # Convert labels to string
    train_df["class"] = train_df["class"].astype(str)
    val_df["class"] = val_df["class"].astype(str)
    
    # Generators
    train_gen = train_datagen.flow_from_dataframe(
        train_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=True,
        seed=42
    )
    
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=False
    )
    
    # class weights
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_labels),
        y=train_labels
    )
    
    class_weights = dict(enumerate(class_weights))
    print("Class weights:", class_weights)
    
    # Build fresh model
    model = build_model()
    
    # Callbacks
    callbacks_fold = [
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            f"best_model_fold_{fold}.keras",  
            monitor="val_loss",
            save_best_only=True
        ),
        ReduceLROnPlateau(       #Reduce learning rate if validation loss stops improving
            monitor='val_loss',
            factor=0.3,
            patience=2,
            min_lr=1e-6,
            verbose=1
        )
    ]
    
    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=25,
        class_weight=class_weights,
        callbacks=callbacks_fold,
        verbose=1
    )
    
    # Store results
    val_acc = max(history.history["val_accuracy"])
    fold_val_accuracies.append(val_acc)
    fold_histories.append(history)


------------Fold 1------------
Found 3477 validated image filenames belonging to 2 classes.
Found 1739 validated image filenames belonging to 2 classes.
Class weights: {0: np.float64(1.9446308724832215), 1: np.float64(0.673054587688734)}
Epoch 1/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - accuracy: 0.7886 - loss: 0.4566 - val_accuracy: 0.7430 - val_loss: 0.9238 - learning_rate: 1.0000e-04
Epoch 2/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.8772 - loss: 0.2894 - val_accuracy: 0.7430 - val_loss: 1.7312 - learning_rate: 1.0000e-04
Epoch 3/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 869ms/step - accuracy: 0.8941 - loss: 0.2493
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
109/109 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.8861 - loss: 0.2621 - val_accuracy: 0.7430 - val_loss: 1.9541 - learning_rate: 1.0000e-04
Epoch 4/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9042 - loss: 0.2300 - val_accuracy: 0.7430 - val_loss: 1.8953 - learn

In [19]:
print("\nK-Fold Results\n")
print("Fold Accuracies:", fold_val_accuracies)
print("Mean Accuracy:", np.mean(fold_val_accuracies))


K-Fold Results

Fold Accuracies: [0.74295574426651, 0.74295574426651, 0.7428078055381775]
Mean Accuracy: 0.7429064313570658
time: 0 ns (started: 2026-03-25 01:28:10 +05:30)
